In [ ]:
# mount the drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd /content/drive/MyDrive/Colab Notebooks/

/content/drive/MyDrive/Colab Notebooks


In [ ]:
!pip install -q transformers sentence-transformers scikit-learn pandas tqdm numpy sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 18.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("indiana_reports_with_projections.csv")

print(f"Raw dataset shape: {df.shape}")
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# ── Preserve ALL rows. For rows where impression is missing,
#    fall back to 'findings'; if both are missing, use empty string.
#    DO NOT drop any rows.
def resolve_text(row):
    imp = row.get('impression', '')
    fin = row.get('findings', '')
    if pd.notna(imp) and str(imp).strip():
        return str(imp).strip()
    if pd.notna(fin) and str(fin).strip():
        return str(fin).strip()
    return ''

df['input_text'] = df.apply(resolve_text, axis=1)
df['text_source'] = df.apply(
    lambda r: 'impression' if (pd.notna(r.get('impression','')) and str(r.get('impression','')).strip())
              else ('findings' if (pd.notna(r.get('findings','')) and str(r.get('findings','')).strip())
              else 'empty'),
    axis=1
)

print(f"\nText source breakdown:")
print(df['text_source'].value_counts())
print(f"\nTotal usable rows (non-empty text): {(df['input_text'] != '').sum()}")
print(f"Total rows retained: {len(df)}")

Raw dataset shape: (3689, 9)

Missing values per column:
uid              0
MeSH             0
Problems         0
image            0
indication      83
comparison    1121
findings       490
impression      29
filename         0
dtype: int64

Duplicate rows: 0

Text source breakdown:
text_source
impression    3660
empty           23
findings         6
Name: count, dtype: int64

Total usable rows (non-empty text): 3666
Total rows retained: 3689


In [ ]:
import torch
from transformers import pipeline, BioGptConfig, BioGptForCausalLM, BioGptTokenizer

# Load configuration and explicitly set tie_word_embeddings=False to silence the warning
model_name = "microsoft/BioGPT"
config = BioGptConfig.from_pretrained(model_name)
config.tie_word_embeddings = False

tokenizer = BioGptTokenizer.from_pretrained(model_name)
model = BioGptForCausalLM.from_pretrained(model_name, config=config)

# Initialize the pipeline with the explicitly configured model
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device= 0 if torch.cuda.is_available() else -1
)

print("BioGPT pipeline loaded with explicit configuration.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

BioGPT pipeline loaded with explicit configuration.


In [ ]:
import re

# ── Pathology terms: if ANY of these appear, the impression is NOT normal.
#    This list targets terms that unambiguously describe acute/active findings.
PATHOLOGY_TERMS = [
    "pneumonia", "consolidation", "effusion", "pulmonary edema", "edema",
    "pneumothorax", "atelectasis", "opacity", "opacification",
    "infiltrate", "infiltration", "mass", "nodule", "malignancy",
    "cardiomegaly", "pleural", "interstitial", "airspace disease",
    "worsening", "progression", "new finding", "acute process",
    "fracture", "displaced", "collapse", "cavitation", "abscess",
    "emphysema", "bronchiectasis", "hilar enlargement", "mediastinal widening",
]

# ── Definitive overall-normal phrases: the ENTIRE impression is a clean bill.
OVERALL_NORMAL = [
    "no acute cardiopulmonary", "no acute pulmonary", "no acute chest",
    "no acute findings", "no acute abnormality", "no acute disease",
    "no acute osseous", "normal chest", "normal radiograph",
    "normal study", "normal examination", "unremarkable",
    "within normal limits", "no evidence of acute",
    "no active disease", "no active pulmonary",
    "no significant findings", "no cardiopulmonary disease",
    "no cardiopulmonary abnormality",
    "no acute intrathoracic", "no acute process",
    "no acute traumatic", "no acute bony",
]

# ── Weaker normal signals — only used when no pathology terms are present.
WEAK_NORMAL = [
    "no findings", "no abnormality", "clear lungs",
    "negative chest", "negative study",
]


def fallback_normal(text: str) -> bool:
    """
    Rule-based determination of whether a radiology impression is 'normal'
    (i.e., no acute or significant findings).

    Strategy:
      1. Presence of any pathology term → return False immediately.
      2. Otherwise, check for strong overall-normal phrases → True.
      3. Fall back to weak-normal phrases in the absence of pathology → True.
      4. Default: False (conservative — ambiguity treated as not-normal).
    """
    if not text or not text.strip():
        return False

    t = text.lower()

    # Step 1: hard-block on pathology
    if any(term in t for term in PATHOLOGY_TERMS):
        return False

    # Step 2: strong overall-normal signal
    if any(phrase in t for phrase in OVERALL_NORMAL):
        return True

    # Step 3: weak normal — only fire if no pathology (already confirmed above)
    if any(phrase in t for phrase in WEAK_NORMAL):
        return True

    return False


In [ ]:
import json as _json


def _safe_list(val) -> list:
    """Ensure a value from JSON parsing is always a list of strings."""
    if isinstance(val, list):
        return [str(v).strip() for v in val if str(v).strip()]
    if isinstance(val, str) and val.strip():
        return [val.strip()]
    return []


def _extract_conditions_rule(text: str) -> list:
    """
    Rule-based condition extraction as fallback when LLM fails.
    Scans for known radiological condition terms in the impression text.
    """
    CONDITION_TERMS = [
        "pneumonia", "consolidation", "pleural effusion", "pulmonary edema",
        "pneumothorax", "atelectasis", "cardiomegaly", "emphysema",
        "bronchiectasis", "interstitial lung disease", "pulmonary fibrosis",
        "mass", "nodule", "fracture", "opacity", "infiltrate",
        "mediastinal widening", "hilar enlargement", "pericardial effusion",
        "aortic dilation", "pulmonary hypertension",
    ]
    t = text.lower()
    found = [term for term in CONDITION_TERMS if term in t]
    return found


def clean_impression(text: str) -> dict:
    """
    Extracts structured fields from a radiology impression using BioGPT.

    BioGPT is a decoder-only model — the prompt is a prefix the model continues.
    The critical 'normal' field is always overridden by the rule-based
    fallback_normal() for reliability. Conditions are also cross-validated
    against rule-based extraction.

    Returns a dict with: normal (bool), conditions (list), findings (list), summary (str).
    Never returns None — always falls back to a safe default.
    """
    if not text or not text.strip():
        return {
            "normal": False,
            "conditions": [],
            "findings": [],
            "summary": ""
        }

    text = text.strip()

    # Completion-style prompt for BioGPT (decoder-only)
    prompt = (
        f"Radiology impression: {text}\n"
        f"Structured JSON with keys normal (bool), conditions (list), "
        f"findings (list), summary (string):\n"
        f"{{"
    )

    parsed = None
    try:
        result = pipe(
            prompt,
            max_new_tokens=150,
            do_sample=False,          # greedy — deterministic output
            pad_token_id=pipe.tokenizer.eos_token_id,
        )[0]["generated_text"]

        # Strip the prompt prefix so we only parse newly generated text
        generated = result[len(prompt):]
        json_candidate = "{" + generated

        # Extract first complete JSON object (non-greedy on inner braces)
        match = re.search(r'\{[^{}]*\}', json_candidate, re.DOTALL)
        if match:
            parsed = _json.loads(match.group())
    except Exception:
        pass  # LLM failed — fallback below handles it

    # ── Validate and sanitize parsed output ──────────────────────────────────
    if parsed and isinstance(parsed, dict):
        conditions_llm = _safe_list(parsed.get("conditions", []))
        findings_llm   = _safe_list(parsed.get("findings", []))
        summary_llm    = str(parsed.get("summary", "")).strip()

        # Merge LLM-extracted conditions with rule-based ones (union, no duplicates)
        conditions_rule = _extract_conditions_rule(text)
        conditions_merged = list(dict.fromkeys(conditions_llm + conditions_rule))

        return {
            "normal":     fallback_normal(text),       # always authoritative
            "conditions": conditions_merged,
            "findings":   findings_llm,
            "summary":    summary_llm if summary_llm else text,
        }

    # ── Full fallback: LLM produced no valid JSON ─────────────────────────────
    return {
        "normal":     fallback_normal(text),
        "conditions": _extract_conditions_rule(text),
        "findings":   [],
        "summary":    text,
    }


In [ ]:
import os
from tqdm import tqdm

CHECKPOINT_PATH = './impression_checkpoint.json'

# ── Resume from checkpoint if one exists (crash recovery) ────────────────────
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r') as f:
        _ckpt = _json.load(f)
    structured_data = _ckpt['data']  # {str(idx): result_dict}
    print(f"Resumed checkpoint: {len(structured_data)} rows already processed.")
else:
    structured_data = {}
    print("Starting fresh.")

errors = []
CHECKPOINT_EVERY = 100

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Structuring impressions"):
    key = str(idx)
    if key in structured_data:
        continue  # already processed in a prior run

    text = row['input_text']  # resolved impression or findings fallback

    try:
        structured_data[key] = clean_impression(text)
    except Exception as e:
        errors.append({'idx': idx, 'error': str(e), 'text_preview': text[:80]})
        # Safe fallback — row is NOT lost
        structured_data[key] = {
            "normal":     fallback_normal(text),
            "conditions": _extract_conditions_rule(text),
            "findings":   [],
            "summary":    text,
        }

    # Checkpoint every N rows
    if (idx + 1) % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_PATH, 'w') as f:
            _json.dump({'data': structured_data}, f)

# Final checkpoint save
with open(CHECKPOINT_PATH, 'w') as f:
    _json.dump({'data': structured_data}, f)

df_full = df.copy()
df_full['structured'] = [structured_data[str(i)] for i in df_full.index]

print(f"\nTotal rows:      {len(df_full)}")
print(f"Rows processed:  {len(structured_data)}")
print(f"Errors (used fallback, no data lost): {len(errors)}")
if errors:
    print("Sample errors:", errors[:3])


Starting fresh.


Structuring impressions:   0%|          | 0/3689 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Structuring impressions:   0%|          | 3/3689 [00:06<1:45:50,  1.72s/it]Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/t


Total rows:      3689
Rows processed:  3689
Errors (used fallback, no data lost): 0


In [ ]:
def safe_get(record, key, default):
    """Safely extract a key from a structured dict; return default on failure."""
    try:
        val = record[key]
        return val if val is not None else default
    except (KeyError, TypeError):
        return default


df_full['summary']    = df_full['structured'].apply(lambda x: safe_get(x, 'summary', ''))
df_full['normal']     = df_full['structured'].apply(lambda x: safe_get(x, 'normal', False))
df_full['conditions'] = df_full['structured'].apply(
    lambda x: ', '.join(safe_get(x, 'conditions', []))
)
df_full['findings_extracted'] = df_full['structured'].apply(
    lambda x: ', '.join(safe_get(x, 'findings', []))
)

# ── Replace empty summaries with input_text so embedding never gets blank strings
mask_empty = df_full['summary'].str.strip() == ''
df_full.loc[mask_empty, 'summary'] = df_full.loc[mask_empty, 'input_text']

print(f"Rows with empty summary after fallback: {(df_full['summary'].str.strip() == '').sum()}")
print(f"Normal (True):  {df_full['normal'].sum()}")
print(f"Normal (False): {(~df_full['normal']).sum()}")
print(f"Rows with extracted conditions: {(df_full['conditions'] != '').sum()}")
print(f"Rows with extracted findings:   {(df_full['findings_extracted'] != '').sum()}")


Rows with empty summary after fallback: 23
Normal (True):  1911
Normal (False): 1778
Rows with extracted conditions: 1056
Rows with extracted findings:   0


In [ ]:
print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)
print(f"Total rows:              {len(df_full)}")
print(f"Text from impression:    {(df_full['text_source'] == 'impression').sum()}")
print(f"Text from findings:      {(df_full['text_source'] == 'findings').sum()}")
print(f"Rows with empty text:    {(df_full['text_source'] == 'empty').sum()}")
print(f"Normal impressions:      {df_full['normal'].sum()} ({df_full['normal'].mean()*100:.1f}%)")
print(f"Abnormal impressions:    {(~df_full['normal']).sum()} ({(~df_full['normal']).mean()*100:.1f}%)")
print(f"\nTop 10 extracted conditions:")
all_conds = df_full['conditions'].str.split(', ').explode()
print(all_conds[all_conds != ''].value_counts().head(10))


DATA QUALITY REPORT
Total rows:              3689
Text from impression:    3660
Text from findings:      6
Rows with empty text:    23
Normal impressions:      1911 (51.8%)
Abnormal impressions:    1778 (48.2%)

Top 10 extracted conditions:
conditions
atelectasis         253
pleural effusion    247
cardiomegaly        224
pneumothorax        142
pneumonia           139
consolidation       125
nodule              120
infiltrate          119
opacity              97
emphysema            90
Name: count, dtype: int64


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")
text_model = AutoModel.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")
device = "cuda" if torch.cuda.is_available() else "cpu"
text_model.to(device).eval()

def pubmedbert_encode(texts, batch_size=64):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = text_model(**inputs)
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        emb = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        all_embeddings.append(emb.cpu().numpy())
    return np.vstack(all_embeddings)

embeddings = pubmedbert_encode(df_full['summary'].tolist())
print(f"Embeddings shape: {embeddings.shape}")  # (n, 768)


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings shape: (3689, 768)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ── Silhouette sweep to validate k=6 (or find a better k) ───────────────────
k_range = range(4, 9)
sil_scores = {}

print("Silhouette scores across k values:")
for k_try in k_range:
    km_try = KMeans(n_clusters=k_try, random_state=42, n_init=10)
    labels_try = km_try.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels_try, sample_size=1000, random_state=42)
    sil_scores[k_try] = score
    print(f"  k={k_try}: silhouette = {score:.4f}")

best_k = max(sil_scores, key=sil_scores.get)
print(f"\nBest k by silhouette: {best_k}")

# ── Use k=6 to stay consistent with existing labels; note if best_k differs
k = 6
if best_k != k:
    print(f"Note: silhouette suggests k={best_k}, but using k={k} for label consistency.")

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df_full['cluster'] = kmeans.fit_predict(embeddings)

print(f"\nCluster size distribution (k={k}):")
print(df_full['cluster'].value_counts().sort_index())
print(f"\nFinal silhouette score: {silhouette_score(embeddings, df_full['cluster']):.4f}")


Silhouette scores across k values:
  k=4: silhouette = 0.2143
  k=5: silhouette = 0.2129
  k=6: silhouette = 0.1711
  k=7: silhouette = 0.1788
  k=8: silhouette = 0.1843

Best k by silhouette: 4
Note: silhouette suggests k=4, but using k=6 for label consistency.

Cluster size distribution (k=6):
cluster
0     551
1     133
2     597
3    1043
4     454
5     911
Name: count, dtype: int64

Final silhouette score: 0.1829


In [ ]:
for i in range(k):
    cluster_df = df_full[df_full['cluster'] == i]
    normal_pct = cluster_df['normal'].mean() * 100
    print(f"\n{'='*55}")
    print(f"CLUSTER {i}  |  n={len(cluster_df)}  |  normal={normal_pct:.1f}%")
    print(f"{'='*55}")

    # Top conditions in this cluster
    cond_series = cluster_df['conditions'].str.split(', ').explode()
    top_conds = cond_series[cond_series != ''].value_counts().head(5)
    if not top_conds.empty:
        print(f"Top conditions: {', '.join(top_conds.index.tolist())}")

    print("Sample summaries:")
    for s in cluster_df['summary'].head(8):
        print(f"  - {s[:100]}")



CLUSTER 0  |  n=551  |  normal=68.2%
Top conditions: emphysema, cardiomegaly, fracture, pneumothorax, nodule
Sample summaries:
  - 
  - Emphysema, however no acute cardiopulmonary finding.
  - No acute cardiopulmonary abnormality..
  - No acute cardiopulmonary abnormalities. .
  - No acute abnormalities are seen. .
  - No acute abnormality. .
  - Clear lungs.
  - 1. No acute radiographic cardiopulmonary process.

CLUSTER 1  |  n=133  |  normal=91.7%
Top conditions: nodule
Sample summaries:
  - Normal chest x-XXXX.
  - Negative preoperative chest x-XXXX.
  - Normal chest.
  - Clear lungs
  - Normal chest
  - Normal chest
  - Unremarkable radiographs of the chest.
  - Normal chest

CLUSTER 2  |  n=597  |  normal=6.7%
Top conditions: atelectasis, cardiomegaly, pleural effusion, infiltrate, pneumonia
Sample summaries:
  - Basilar atelectasis. No confluent lobar consolidation or pleural effusion.
  - Borderline enlargement of the cardiac silhouette without acute pulmonary disease.
  - Hype

In [ ]:
cluster_to_label = {
    0: 'Mild Negative',
    1: 'Pulmonary Disease',
    2: 'Normal',
    3: 'Normal (Explicit)',
    4: 'Chronic / Mixed',
    5: 'Cardiac / Structural'
}

df_full['final_label'] = df_full['cluster'].map(cluster_to_label)

# Sanity check: unmapped clusters produce NaN labels
unmapped = df_full['final_label'].isnull().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} rows have unmapped cluster IDs — extend cluster_to_label.")
else:
    print("All clusters mapped successfully.")

# Cross-validate: 'Normal' and 'Normal (Explicit)' clusters should have high normal=True rates
print("\nNormal-flag rate per label (sanity check):")
print(df_full.groupby('final_label')['normal'].mean().sort_values(ascending=False).round(3))


All clusters mapped successfully.

Normal-flag rate per label (sanity check):
final_label
Pulmonary Disease       0.917
Normal (Explicit)       0.883
Mild Negative           0.682
Chronic / Mixed         0.610
Cardiac / Structural    0.192
Normal                  0.067
Name: normal, dtype: float64


In [ ]:
print("Label distribution (proportion):")
print(df_full['final_label'].value_counts(normalize=True).round(3))
print("\nLabel distribution (absolute counts):")
print(df_full['final_label'].value_counts())


Label distribution (proportion):
final_label
Normal (Explicit)       0.283
Cardiac / Structural    0.247
Normal                  0.162
Mild Negative           0.149
Chronic / Mixed         0.123
Pulmonary Disease       0.036
Name: proportion, dtype: float64

Label distribution (absolute counts):
final_label
Normal (Explicit)       1043
Cardiac / Structural     911
Normal                   597
Mild Negative            551
Chronic / Mixed          454
Pulmonary Disease        133
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    df_full['final_label'],
    test_size=0.2,
    random_state=42,
    stratify=df_full['final_label']  # preserve class balance in splits
)

clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(f"Accuracy: {clf.score(X_test, y_test):.4f}")
print("\nFull Classification Report:")
print(classification_report(y_test, y_pred, digits=3))
print("\nConfusion Matrix:")
labels = sorted(df_full['final_label'].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)


Accuracy: 0.9864

Full Classification Report:
                      precision    recall  f1-score   support

Cardiac / Structural      0.978     0.973     0.975       182
     Chronic / Mixed      1.000     1.000     1.000        91
       Mild Negative      1.000     0.991     0.995       110
              Normal      0.958     0.966     0.962       119
   Normal (Explicit)      0.995     1.000     0.998       209
   Pulmonary Disease      1.000     1.000     1.000        27

            accuracy                          0.986       738
           macro avg      0.989     0.988     0.988       738
        weighted avg      0.986     0.986     0.986       738


Confusion Matrix:
                      Cardiac / Structural  Chronic / Mixed  Mild Negative  \
Cardiac / Structural                   177                0              0   
Chronic / Mixed                          0               91              0   
Mild Negative                            0                0            109   


In [ ]:
def predict(text: str) -> str:
    """
    End-to-end prediction: raw impression text → final_label.
    clean_impression always returns a dict (never None), so no None check needed.
    """
    structured = clean_impression(text)
    summary = structured.get('summary') or text

    emb = pubmedbert_encode([summary])

    return clf.predict(emb)[0]


# ── Tests ─────────────────────────────────────────────────────────────────────
test_cases = [
    "No acute cardiopulmonary abnormality.",
    "Left lung opacity suggesting pneumonia with small pleural effusion.",
    "Mild cardiomegaly. No acute pulmonary edema.",
    "Normal chest radiograph.",
    "Bilateral lower lobe atelectasis. Stable chronic interstitial changes.",
]

print("Predictions on test impressions:")
for t in test_cases:
    print(f"  [{predict(t):<22}]  {t}")


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predictions on test impressions:


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Normal (Explicit)     ]  No acute cardiopulmonary abnormality.


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Normal                ]  Left lung opacity suggesting pneumonia with small pleural effusion.


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Normal                ]  Mild cardiomegaly. No acute pulmonary edema.
  [Pulmonary Disease     ]  Normal chest radiograph.
  [Normal                ]  Bilateral lower lobe atelectasis. Stable chronic interstitial changes.


In [ ]:
output_path = './cluster_categorized_data.csv'
df_full.to_csv(output_path, index=False)

# Verify the written file matches in-memory data
check = pd.read_csv(output_path)
assert len(check) == len(df_full), \
    f"Row count mismatch: wrote {len(df_full)}, read back {len(check)}"
assert list(check.columns) == list(df_full.columns), "Column mismatch on save."

print(f"Saved {len(df_full)} rows to '{output_path}'")
print(f"Columns: {list(df_full.columns)}")
print(f"File verification: PASSED")


Saved 3689 rows to './cluster_categorized_data.csv'
Columns: ['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression', 'filename', 'input_text', 'text_source', 'structured', 'summary', 'normal', 'conditions', 'findings_extracted', 'cluster', 'final_label']
File verification: PASSED
